# Day 1 — Data and Mathematical Foundation

## Goal

Understand how customer data becomes mathematical input for Logistic Regression.

## Core Idea

X = input features

y = target

For one customer:

X = [x₁, x₂, ..., xₙ]

The linear model calculates:

z = wᵀX + b

Where:

- X = feature vector
- w = weight vector
- b = bias
- z = linear combination

In [1]:
import pandas as pd

df = pd.read_csv("../data/dataset.csv")

df.head()

,age,tenure,monthly_charges,support_calls,contract_type,churn
0,22,2,75,5,month-to-month,1
1,45,36,55,1,two-year,0
2,31,8,90,4,month-to-month,1
3,52,48,60,1,two-year,0
4,28,5,85,3,month-to-month,1


In [2]:


X = df.drop("churn", axis=1)
y = df["churn"]

print("Features (X):")
print(X)

print("\nTarget (y):")
print(y)

Features (X):
    age  tenure  monthly_charges  support_calls   contract_type
0    22       2               75              5  month-to-month
1    45      36               55              1        two-year
2    31       8               90              4  month-to-month
3    52      48               60              1        two-year
4    28       5               85              3  month-to-month
5    39      24               65              2        one-year
6    61      60               50              0        two-year
7    25       4               95              6  month-to-month
8    47      30               58              1        one-year
9    34      12               88              4  month-to-month
10   56      42               62              1        two-year
11   29       7               78              5  month-to-month
12   41      18               70              2        one-year
13   63      55               52              0        two-year
14   36      10           

In [3]:
x = [22, 2, 75, 5]
w = [0.1, 0.2, 0.05, 0.3]
b = 1

z = (
    w[0] * x[0]
    + w[1] * x[1]
    + w[2] * x[2]
    + w[3] * x[3]
    + b
)

print("z =", z)

z = 8.85


In [4]:
import numpy as np

x = np.array([22, 2, 75, 5])
w = np.array([0.1, 0.2, 0.05, 0.3])
b = 1

z = np.dot(w, x) + b

print("wᵀx + b =", z)

wᵀx + b = 8.85


## Linear Model

For one customer, our numerical feature vector is:

x = [x₁, x₂, ..., xₙ]

The model has a weight vector:

w = [w₁, w₂, ..., wₙ]

The linear model calculates:

z = wᵀx + b

The dot product is:

wᵀx = w₁x₁ + w₂x₂ + ... + wₙxₙ

For our example:

x = [22, 2, 75, 5]

w = [0.1, 0.2, 0.05, 0.3]

b = 1

Therefore:

z = (0.1×22) + (0.2×2) + (0.05×75) + (0.3×5) + 1

z = 8.85

### Important

The model has not learned these weights yet.

We selected them manually only to understand the mathematics.

Later, gradient descent will learn the weights from the data.

## Numerical vs Categorical Features

Our dataset contains two types of features.

### Numerical features

These already have numerical values:

- age
- tenure
- monthly_charges
- support_calls

Example:

x = [22, 2, 75, 5]

These can directly participate in mathematical operations.

### Categorical feature

contract_type contains categories:

- month-to-month
- one-year
- two-year

These cannot directly be multiplied by numerical weights.

Therefore, we will transform the categorical feature into a numerical representation.

We will learn and implement this encoding during Day 2.

### Important

We should not simply assign:

month-to-month = 1
one-year = 2
two-year = 3

without understanding the consequences, because those numbers could incorrectly suggest an ordered relationship.


## Dimensions of X and w

Our dataset contains:

- 20 customers
- 4 numerical features used for today's mathematical example

Therefore:

X ∈ R^(20×4)

For one customer:

x ∈ R^4

The weight vector contains one weight for every feature:

w ∈ R^4

Therefore the dot product is valid:

wᵀx

because both x and w contain 4 corresponding values.

The linear model is:

z = wᵀx + b

The model has one weight for each input feature.

In [5]:
print("X shape:", X.shape)

x_numeric = X[["age", "tenure", "monthly_charges", "support_calls"]].iloc[0].to_numpy()

print("One customer x:", x_numeric)
print("Number of features:", len(x_numeric))

X shape: (20, 5)
One customer x: [22  2 75  5]
Number of features: 4


In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import load_data

dataset_path = PROJECT_ROOT / "data" / "dataset.csv"

df_test = load_data(dataset_path)

print("Shape:", df_test.shape)
print(df_test.head())

Shape: (20, 6)
   age  tenure  monthly_charges  support_calls   contract_type  churn
0   22       2               75              5  month-to-month      1
1   45      36               55              1        two-year      0
2   31       8               90              4  month-to-month      1
3   52      48               60              1        two-year      0
4   28       5               85              3  month-to-month      1


# Day 2 — Preprocessing and Logistic Regression Mathematics


## Goal

Transform raw dataset features into standardized numerical inputs, then apply the Sigmoid function to output probabilities.


## Core Idea

Machine learning models only understand numbers, and they learn best when those numbers are on a similar scale.

->Train/Test Split: Isolate a test set to evaluate real-world performance later.

->Encoding: Convert text categories (like "month-to-month") into 0s and 1s.

->Standardization: x−μ/σ

->Sigmoid: σ(z)= 1/1+e^(−z)

In [7]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Import the functions you wrote!
from src.preprocessing import (
    load_data, train_test_split, encode_categorical, 
    calculate_mean, calculate_std, standardize
)
from src.logistic_regression import (
    sigmoid, predict_probability, probability_to_class
)

dataset_path = PROJECT_ROOT / "data" / "dataset.csv"
df = load_data(dataset_path)

X = df.drop("churn", axis=1)
y = df["churn"]

## 1. Train / Test Split

If a student sees the exam paper before the test, getting a 100% doesn't mean they are smart—it just means they memorized it! The same applies to Machine Learning. We must hide some data (the test set) so we can evaluate the model fairly later.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

Training features shape: (16, 5)
Testing features shape: (4, 5)


## 2. Categorical Encoding (One-Hot Encoding)
In Day 1, we saw that contract_type contains text like "month-to-month" or "two-year". You cannot multiply a weight w 
i
​	
  by "month-to-month"!

We could just assign 1, 2, and 3 to them. But the model would think that "two-year" (3) is mathematically greater or worth more than "month-to-month" (1). Instead, we use your encode_categorical function to create binary flags.

In [9]:
X_train_encoded, X_test_encoded = encode_categorical(X_train, X_test)

print("Original Data (First 2 rows):")
print(X_train[["age", "contract_type"]].head(2))

print("\nEncoded Data (First 2 rows):")
print(X_train_encoded[["age", "month-to-month", "one-year", "two-year"]].head(2))

Original Data (First 2 rows):
    age contract_type
12   41      one-year
10   56      two-year

Encoded Data (First 2 rows):
    age  month-to-month  one-year  two-year
12   41               0         1         0
10   56               0         0         1


## 3. Feature Standardization
Right now, age might be around 40, but monthly_charges might be 90. If we do not scale these, the gradients for larger features will bounce around wildly during gradient descent.

We standardize them using the training mean μ and standard deviation σ:

x(scaled) = x−μ / σ
​	

Important: We must calculate the mean and standard deviation strictly on the training set, and use those exact same values to scale the test set!

In [10]:
# Convert to NumPy for mathematical operations
X_train_num = X_train_encoded.to_numpy(dtype=float)
X_test_num = X_test_encoded.to_numpy(dtype=float)
y_train_num = y_train.to_numpy(dtype=int)

# Calculate mu and sigma on training data
train_mean = calculate_mean(X_train_num)
train_std = calculate_std(X_train_num, train_mean)

# Scale both sets
X_train_scaled = standardize(X_train_num, train_mean, train_std)
X_test_scaled = standardize(X_test_num, train_mean, train_std)

print("First customer BEFORE scaling:")
print(np.round(X_train_num[0], 2))

print("\nFirst customer AFTER scaling:")
print(np.round(X_train_scaled[0], 2))

First customer BEFORE scaling:
[41. 18. 70.  2.  0.  1.  0.]

First customer AFTER scaling:
[-0.1  -0.47  0.08 -0.17 -0.77  1.73 -0.77]


## 4. The Sigmoid Function

We use the Sigmoid function, which smoothly squishes any number between −∞ and +∞ into a range between 0 and 1:

In [11]:
sample_z = np.array([-5, 0, 8.85])
print("z values:", sample_z)
print("Probabilities:", np.round(sigmoid(sample_z), 4))

z values: [-5.    0.    8.85]
Probabilities: [0.0067 0.5    0.9999]


## 5. Putting it all together: Making Predictions

Now, let's use all your functions together. We will set up some dummy weights (since we haven't learned Gradient Descent yet), calculate the probabilities, and use probability_to_class function to make the final decision!

In [12]:
# We now have 7 features (age, tenure, charges, calls, and 3 contract types)
w_dummy = np.array([0.1, -0.4, 0.2, 0.5, 0.3, -0.2, -0.5])
b_dummy = 0.0

# 1. Calculate probabilities for the first 5 training customers
probs = predict_probability(X_train_scaled[:5], w_dummy, b_dummy)

# 2. Convert probabilities to final class predictions (threshold = 0.5)
predictions = probability_to_class(probs, threshold=0.5)

print("Customer | Probability | Prediction | Actual")
print("-" * 46)
for i in range(5):
    p = probs[i]
    pred = predictions[i]
    actual = y_train_num[i]
    print(f"   {i}     |    {p:.4f}   |      {pred}     |   {actual}")

Customer | Probability | Prediction | Actual
----------------------------------------------
   0     |    0.4789   |      0     |   0
   1     |    0.1927   |      0     |   0
   2     |    0.0980   |      0     |   0
   3     |    0.1547   |      0     |   0
   4     |    0.1655   |      0     |   0
